# Lesson 6 — Confidence, escalation thresholds, and a claim that doesn't hold up

Lessons 1–5 covered the primitives and how to compose them. This lesson is about what to do with the answers once you have them: when to act automatically, when to double-check, and when to hand off to a person. It also closes a loop that's been open since lesson 1 — the vendor's "0% hallucination" marketing claim versus what the technical docs actually promise.

Ask your agent if anything's unclear.

<details>
  <summary><strong>Primary source:</strong> docs.typesafe.ai/confidence (click to expand)</summary>

# Confidence

> How TypeSafe reports certainty, how it differs from probability, and how to use it to control system behavior.

All Score and Choice answers from TypeSafe include a `probabilities` property representing the probability distribution across the options (for Choice) or levels (for Score). The *shape* of that distribution is what tells you how certain the model is: concentrated on one outcome means a confident answer, spread out means an uncertain one.

The answer's `confidence` property collapses that shape into a single number from 0 to 1, so you can threshold on it without doing the math yourself. (Noul answers don't carry one.)

## Confidence is derived from the probabilities

`confidence` is a statistic computed from the probability distribution the answer already gives you. TypeSafe computes it for you and returns it on every Choice and Score answer, so the common case needs no extra work on your side.

> **A solid default:** We provide `confidence` as a convenient measure that fits most use-cases, but you are never locked into our definition. Depending on what you are evaluating, a different measure may serve you better, which is exactly why we give you the full `probabilities` in the response. The pros and cons of different computations is a specialized topic that we'll keep to a separate cookbook rather than this page, and will add the link here when we do!

For a [Choice](https://docs.typesafe.ai/primitives/choice), the distribution is `probabilities` across your options. For a [Score](https://docs.typesafe.ai/primitives/score), it is the distribution across your levels. In both cases a flatter distribution means lower confidence: low confidence on a Choice often means none of the options are a clear winner over the others, and low confidence on a Score often means the levels are ambiguous, multi-dimensional, or the state doesn't contain enough to go on.

## "I don't know" is a useful signal

If an intelligent system, whether human or machine, cannot express honest uncertainty, the system cannot be trusted.

Confidence gives you a built-in mechanism for the model to say "I'm not sure about this one." This lets your code implement different behavior for different levels of certainty, which is the foundation for building systems you can actually rely on.

## Three paths for using confidence in your code

A useful starting pattern is to divide confidence into three ranges, each producing a different system behavior:

**High confidence:** Act automatically. The model has a clear read and you can proceed without human involvement.

**Medium confidence:** Proceed with caution. The model has a reasonable answer but is not certain. Depending on context, you might ask the user to confirm, flag for review, or gather more information before acting.

**Low confidence:** Do not act. Route to a human, request clarification, or fall back to a different system. The model is telling you it does not have enough information or the question is not a good fit.

Where you draw those boundaries depends on the stakes.

## Thresholds scale with risk

A confidence threshold is not one number. Different actions within the same system should be gated at different levels depending on the consequences of getting it wrong.

```python
response = client.system_one(
    state=user_message,
    questions={
        "action": Choice(
            instructions="What is the user trying to do?",
            criteria={
                "check_balance": "View account balance",
                "approve_transfer": "Approve the pending withdrawal request",
                "support": "Get help with an issue",
            },
        ),
    },
)

action = response.answers["action"]
confidence = action.confidence

if confidence < 0.5:
    # Model is genuinely unsure. Don't guess.
    route_to_human(user_message)

elif action.choice == "check_balance":
    # Low stakes. Showing the wrong screen is recoverable.
    show_balance(account_id)

elif action.choice == "approve_transfer":
    if confidence > 0.9:
        # High stakes, high confidence. Proceed with confirmation.
        confirm_then_execute(account_id)
    else:
        # High stakes, moderate confidence. Verify first.
        ask_user_to_confirm(account_id)
```

The 0.5 confidence floor catches anything the model reports as genuinely uncertain. Above that, the threshold for acting without confirmation is higher for a destructive operation than for a read-only one. Your code encodes the risk tolerance.

> **Note:** The correct threshold values depend on your domain and the performance of the model for your use case. Start with conservative thresholds, test with your own data, and adjust as you observe results.

</details>

## The core idea

Nothing here is a new primitive — it's the discipline for using `confidence`, which you've had since lesson 1's `Choice` answers. Two things this page adds on top of what we've already used:

1. **A named three-tier pattern**: act automatically / proceed with caution / don't act. You've effectively used the "don't act" end of this since lesson 5 (routing to a human when nothing fits), but this is the first time it's a deliberate framework rather than a one-off `if`.
2. **Thresholds scale with the stakes of being wrong, not with a fixed number.** The docs' own example uses one floor (`0.5`) to catch outright uncertainty, then a *second*, higher bar (`0.9`) specifically for the destructive action — the read-only action gets no confidence gate at all, because being wrong there costs nothing.

In [ ]:
import { client } from "../assets/typesafe-client.ts";
import { choice } from "npm:@typesafe-ai/sdk";

const userMessage = "can you approve the $5,000 withdrawal I requested yesterday?";

const response = await client.systemOne({
  state: userMessage,
  questions: {
    action: choice("What is the user trying to do?", {
      checkBalance: "View account balance",
      approveTransfer: "Approve the pending withdrawal request",
      support: "Get help with an issue",
    }),
  },
});

const { action } = response.answers;
const confidence = action.confidence;

console.log("action:", action.choice, "confidence:", confidence.toFixed(2));

if (confidence < 0.5) {
  console.log("→ genuinely unsure, route to a human");
} else if (action.choice === "checkBalance") {
  console.log("→ low stakes, show the balance");
} else if (action.choice === "approveTransfer") {
  console.log(
    confidence > 0.9
      ? "→ high stakes, high confidence: proceed with confirmation"
      : "→ high stakes, moderate confidence: verify with the user first",
  );
} else {
  console.log("→ route to support");
}

## What just happened

One `Choice`, one number, three possible code paths — and notice `checkBalance` doesn't check `confidence` at all. That's deliberate, straight from the docs' own example: a wrong guess there is cheap to recover from, so gating it on confidence would just be friction.

**Try it yourself:** predict the action and roughly how confident the model will be for the message above (it's fairly explicit) before running it. Then try a genuinely ambiguous message like "what's going on with my account" and predict whether it clears the `0.9` bar, lands in "verify first," or drops below `0.5` entirely.

## The marketing claim, revisited

Back in lesson 1 we flagged a discrepancy from the research behind this series: typesafe.ai's marketing site claims a **"0% hallucination rate," "mathematically guaranteed through structured outputs."** Neither `system-one.md` nor this `confidence.md` page makes that claim — and this page comes close to saying the opposite:

> "The correct threshold values depend on your domain and the performance of the model for your use case. Start with conservative thresholds, test with your own data, and adjust as you observe results."

That's an explicit acknowledgment that the model can be wrong at any confidence level, and that *you* are responsible for finding thresholds that hold up on your own data. If answers were mathematically guaranteed to be 0% wrong, there would be nothing to threshold, test, or adjust against — you'd just always act on the answer, no `confidence` field required.

What's actually guaranteed — and it's a real, defensible guarantee — is the answer's **shape**. A `Choice` answer will always be one of your defined options; it cannot come back as a fourth option, a paragraph of prose, or malformed JSON. That's what "structured output" buys you: not that the pick is *correct*, only that it's *well-formed*. `confidence` and the threshold pattern this lesson just walked through are the tools for managing the gap between "well-formed" and "correct." If the two were the same thing, there would be no reason for either to exist.

## Next up

That's the core arc this series set out to cover: fundamentals (System One, State, Choice), the other primitives (Score, Noul), composition patterns (fan-out, composite scoring, function calling), and now confidence — including the one place the vendor's own pitch doesn't survive contact with its own docs.

What's left is deep cuts (reranking, hierarchical classification, citation checks) rather than required next steps. The real next decision is whether to keep going deeper on one of those, or start assembling the talk itself from lessons 1–6.

Questions? Ask your agent — it has the same research this lesson was built from, in `research/typesafe-ai.md`.